# N06 · Activation Recompute：用计算换显存的边界在哪里？


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

Activation checkpointing / recompute 常被一句话概括为“用计算换显存”。但工程上要问得更细：哪些激活不保存？backward 何时重算？会不会影响随机性？为什么 backward 变慢却可能整体更好？什么时候开了也没用？


## 学习地图与版本说明（截至 2026-04-30）

本节要把 activation checkpointing 从口号拆成机制：普通 autograd 为 backward 保存中间激活；checkpoint 区域选择不保存一部分激活，backward 时重新执行 forward 片段来恢复需要的张量。因此它不是“免费省显存”，而是把显存压力转移成额外计算、调度和可能的 RNG 管理成本。

版本上，本教程优先参考 PyTorch stable 的 `torch.utils.checkpoint` 文档。当前文档要求显式传入 `use_reentrant`，并推荐理解 reentrant 与 non-reentrant 的差异；non-reentrant 支持更多 autograd 用法，并可通过 early stop 在重算到所需激活后停止。文档也提醒 dropout 等随机操作会涉及 RNG state 保存与恢复，函数副作用可能导致前后不一致。

学完本节，你应该能判断 checkpointing 是否对当前 OOM 有用：如果模型加载就 OOM，它通常帮不上；如果峰值来自长序列激活，它可能很有效；如果瓶颈已经是 compute-bound，盲目加 checkpoint 可能让 tokens/sec 明显下降。真正的优化要看“是否让原本跑不动的配置跑起来”以及“单位资源吞吐是否更好”。


## 1. Autograd 为什么需要保存激活？

反向传播计算梯度时，很多公式需要 forward 的中间值。例如 Linear 的权重梯度需要输入激活，LayerNorm backward 需要均值/方差或输入相关信息。PyTorch autograd 会为 backward 保存必要 tensor。

如果层数多、序列长、batch 大，保存的中间激活会成为显存大头。Activation checkpointing 的做法是：forward 时少保存一部分中间激活；backward 需要时重新跑对应 forward 片段，把中间值再算出来。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def recompute_tradeoff(layers, segment_size, base_activation_per_layer=0.25, base_compute_per_layer=1.0):
    segments = max(1, (layers + segment_size - 1) // segment_size)
    saved_activation = segments * base_activation_per_layer  # 简化：每段只保存边界
    no_ckpt_activation = layers * base_activation_per_layer
    extra_compute = layers * base_compute_per_layer * (1 + 0.65)  # backward 中部分重算，教学估计
    return {"layers": layers, "segment_size": segment_size, "保存激活GB": saved_activation, "不开重算激活GB": no_ckpt_activation, "相对计算": extra_compute / (layers * base_compute_per_layer)}

rows = [recompute_tradeoff(32, s) for s in [1, 2, 4, 8, 16, 32]]
df = pd.DataFrame(rows)
display(df.round(2))
df.plot(x="segment_size", y=["保存激活GB", "不开重算激活GB"], marker="o", title="checkpoint segment 对激活保存量的影响")
plt.show()


## 2. PyTorch checkpoint 的实践细节

PyTorch `torch.utils.checkpoint` 官方文档明确：checkpointing 是 compute-memory trade-off。新版 PyTorch 推荐显式传 `use_reentrant`，并提供非 reentrant 版本、determinism check、debug 等参数。

工程注意点：

- checkpoint 包住的函数在 backward 会重新执行，函数里不要有不可重复副作用。
- dropout 等随机操作需要处理 RNG state，否则 forward/backward 重算可能不一致。
- checkpoint 的边界太细会增加调度开销；太粗则省显存有限。
- 开 checkpoint 后 backward 变慢是预期，不一定是回归。


In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.checkpoint import checkpoint

    class Block(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(128, 512), nn.GELU(), nn.Linear(512, 128))
        def forward(self, x):
            return self.net(x)

    block = Block()
    x = torch.randn(16, 128, requires_grad=True)
    y = checkpoint(block, x, use_reentrant=False).sum()
    y.backward()
    print("checkpoint backward ok; grad norm:", x.grad.norm().item())
except Exception as exc:
    print("跳过 torch checkpoint smoke：", exc)


## 3. 什么时候 recompute 有用，什么时候没用？

有用：

- OOM 主要来自激活峰值。
- 你愿意用更多计算换更长 seq 或更大 batch。
- GPU 计算还有余量，而显存是硬瓶颈。

没那么有用：

- 模型加载时就 OOM（参数显存问题）。
- optimizer state OOM（优化器状态问题）。
- 数据/CPU/网络是瓶颈，GPU 算力已经不是主问题。
- 重算导致 step time 大幅上升，但释放的显存没有用于增加 batch/seq。

核心判断：**省下的显存有没有换成有效训练规模或稳定性？** 如果只是省了显存但吞吐下降、batch 不变，未必值得。


## 4. 与 TP/PP/FSDP 的关系

Activation recompute 经常和其他并行策略叠加：

- TP 降低每 rank 的部分权重/激活，但增加通信。
- PP 切层，改变每个 stage 的激活驻留。
- FSDP/ZeRO shard 参数/梯度/优化器状态，但激活仍可能很大。
- Sequence parallel/selective recompute 会更细粒度地减少特定激活保存。

所以在 Megatron/TorchTitan/SLiME 里，recompute 不是孤立开关；它是显存预算的一部分。


## 5. 与本课程的连接

- N01 先估算激活显存。
- L01 可以用 `activation_checkpointing` 配置观察 step time 和 peak memory。
- L05 会讨论 `mgt_recompute_tradeoff_007`：开 recompute 后吞吐下降是否值得。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：activation checkpointing 为什么会让 backward 变慢？

**答案解析：** 因为 backward 需要重新执行部分 forward 来恢复中间激活，相当于用额外计算换显存。

### 题 2：开了 checkpoint 后 peak memory 下降，但 tokens/sec 也下降，算优化成功吗？

**答案解析：** 不一定。若省下显存让你能跑更长 seq、更大 batch 或避免 OOM，可能成功；若训练规模不变且吞吐下降，可能不值得。

### 题 3：模型加载就 OOM，checkpointing 能解决吗？

**答案解析：** 基本不能。模型加载 OOM 是参数显存问题，checkpointing 主要减少训练中保存的激活。

### 题 4：checkpoint 包住含 dropout 的模块有什么注意点？

**答案解析：** backward 重算 forward 时要保证随机性一致，依赖 RNG state 管理。否则梯度可能不对应原 forward。

### 题 5：为什么 selective activation recompute 可能比全量 recompute 更好？

**答案解析：** 不同激活的显存收益和重算成本不同。选择高显存、低重算成本的部分重算，可能得到更好的 trade-off。


## 参考资料

- PyTorch `torch.utils.checkpoint`: https://docs.pytorch.org/docs/stable/checkpoint.html
- PyTorch activation checkpointing blog: https://pytorch.org/blog/activation-checkpointing-techniques/
- Megatron Core model parallel config: https://docs.nvidia.com/megatron-core/developer-guide/latest/apidocs/core/core.model_parallel_config.html
